# Analiza trendów sprzedaży

Celem tego notebooka jest bardziej szczegółowa analiza trendów sprzedaży w sklepie internetowym.

Zakres analizy obejmuje:

- zmiany przychodu w czasie
- liczbę zamówień w czasie
- analizę sprzedaży według dni tygodnia
- analizę statusów zamówień
- analizę sprzedaży według kategorii produktów

Wyniki tej analizy pozwolą lepiej zrozumieć dynamikę sprzedaży i strukturę przychodów.

## Import bibliotek

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

## Wczytanie danych

In [ ]:
df = pd.read_csv("../data/processed/sales_cleaned.csv")

df.head()

## Przygotowanie danych czasowych

Dodajemy pomocnicze kolumny związane z datą zamówienia, które będą wykorzystane w analizie trendów.

In [ ]:
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])

df["purchase_year"] = df["order_purchase_timestamp"].dt.year
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_day"] = df["order_purchase_timestamp"].dt.day
df["day_of_week"] = df["order_purchase_timestamp"].dt.day_name()

## Sprzedaż miesięczna

In [ ]:
monthly_sales = (
    df.groupby("order_month")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .reset_index()
)

monthly_sales.head()

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_sales,
    x="order_month",
    y="revenue"
)

plt.xticks(rotation=45)
plt.title("Przychód miesięczny")
plt.tight_layout()

plt.savefig("../reports/figures/monthly_revenue_trend.png")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=monthly_sales,
    x="order_month",
    y="orders"
)

plt.xticks(rotation=45)
plt.title("Liczba zamówień w czasie")
plt.tight_layout()

plt.savefig("../reports/figures/monthly_orders_trend.png")
plt.show()

### Interpretacja

Analiza miesięczna pozwala ocenić, czy sprzedaż rośnie, spada lub wykazuje sezonowość.

Porównanie przychodu i liczby zamówień pomaga sprawdzić, czy wzrost przychodów wynika z większej liczby zamówień, czy z wyższej wartości koszyka.

## Sprzedaż według dni tygodnia

In [ ]:
day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

sales_by_day = (
    df.groupby("day_of_week")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .reindex(day_order)
    .reset_index()
)

sales_by_day

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=sales_by_day,
    x="day_of_week",
    y="revenue"
)

plt.title("Przychód według dni tygodnia")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("../reports/figures/revenue_by_day_of_week.png")
plt.show()

### Interpretacja

Analiza dni tygodnia pozwala określić, w które dni klienci generują najwyższy przychód.

Tego typu insight może być przydatny przy planowaniu promocji i kampanii marketingowych.

## Analiza statusów zamówień

In [ ]:
order_status_summary = (
    df.groupby("order_status")
    .agg(
        orders=("order_id", "nunique"),
        revenue=("revenue", "sum")
    )
    .sort_values("orders", ascending=False)
    .reset_index()
)

order_status_summary

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=order_status_summary,
    x="order_status",
    y="orders"
)

plt.title("Liczba zamówień według statusu")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig("../reports/figures/orders_by_status.png")
plt.show()

### Interpretacja

Statusy zamówień pokazują, jaka część procesu sprzedaży kończy się sukcesem, a jaka obejmuje zamówienia anulowane lub niedostarczone.

To ważne z perspektywy jakości operacyjnej sklepu.

## Sprzedaż według kategorii produktów

In [ ]:
category_sales = (
    df.groupby("product_category_name")
    .agg(
        revenue=("revenue", "sum"),
        orders=("order_id", "nunique")
    )
    .sort_values("revenue", ascending=False)
    .head(10)
    .reset_index()
)

category_sales

In [ ]:
plt.figure(figsize=(12, 6))

sns.barplot(
    data=category_sales,
    x="revenue",
    y="product_category_name"
)

plt.title("Top 10 kategorii produktów według przychodu")
plt.tight_layout()

plt.savefig("../reports/figures/top_categories_by_revenue.png")
plt.show()

### Interpretacja

Analiza kategorii produktów pozwala wskazać segmenty oferty, które generują największy przychód.

Może to wspierać decyzje dotyczące rozwoju asortymentu i priorytetów sprzedażowych.

In [ ]:
heatmap_data = (
    df.groupby(["purchase_year", "purchase_month"])
    .agg(revenue=("revenue", "sum"))
    .reset_index()
    .pivot(index="purchase_year", columns="purchase_month", values="revenue")
)

heatmap_data

In [ ]:
plt.figure(figsize=(12, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="Blues"
)

plt.title("Heatmapa przychodu według roku i miesiąca")
plt.tight_layout()

plt.savefig("../reports/figures/revenue_heatmap.png")
plt.show()

## Podsumowanie

Analiza trendów sprzedaży pozwoliła zidentyfikować:

- zmiany przychodów i liczby zamówień w czasie
- różnice sprzedaży między dniami tygodnia
- strukturę statusów zamówień
- kategorie produktów generujące najwyższy przychód

W kolejnym etapie przeprowadzona zostanie bardziej szczegółowa analiza produktów.